In [ ]:
# Install ALL required LangChain components at once
!pip install -q streamlit langchain langchain-groq langchain-huggingface langchain-chroma pypdf sentence-transformers langchain-community

# Install the tunnel tool
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-sdk~=1.38.0,

In [8]:
!pip install -q streamlit langchain langchain-groq langchain-huggingface langchain-chroma pypdf sentence-transformers langchain-community pyngrok

In [ ]:
%%writefile app.py
import os
import streamlit as st
import asyncio
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage
from langchain_core.documents import Document

# --- 1. CONFIGURATION ---
# Task 2.4: Model Disclosure
os.environ["GROQ_API_KEY"] = st.sidebar.text_input("Enter Groq API Key", type="password") # <--- PASTE YOUR KEY HERE
os.environ["HF_TOKEN"] = st.sidebar.text_input("Enter Hugging Face Token", type="password") # Optional
RETRIEVER_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GENERATOR_MODEL = "llama-3.1-8b-instant"

st.set_page_config(page_title="Chapter 7 AI Assistant", layout="centered")
st.title("🤖 Chapter 7: Contextual RAG Chatbot")
st.caption(f"Retriever: {RETRIEVER_MODEL} | Generator: {GENERATOR_MODEL}")

# --- 2. INITIALIZE MODELS ---
@st.cache_resource
def init_models():
    llm = ChatGroq(model_name=GENERATOR_MODEL, temperature=0)
    embeddings = HuggingFaceEmbeddings(model_name=RETRIEVER_MODEL)
    return llm, embeddings

llm, embeddings = init_models()

# --- 3. CONTEXTUAL RETRIEVAL LOGIC (Task 2.2) ---
async def enrich_chunk(content, full_text):
    """Adds a situational prefix to chunks to improve retrieval accuracy"""
    prompt = f"Provide a 1-sentence context for this chunk within Chapter 7 (LLMs): {content[:300]}"
    try:
        res = await llm.ainvoke([HumanMessage(content=prompt)])
        return f"Context: {res.content.strip()}\n\n{content}"
    except:
        return content

@st.cache_resource
def build_index():
    if not os.path.exists("7.pdf"):
        st.error("Please upload '7.pdf' to the Colab sidebar!")
        st.stop()

    loader = PyPDFLoader("7.pdf")
    docs = loader.load()
    full_text = " ".join([d.page_content for d in docs])

    # Task 2.1: Chunking strategy
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=80)
    chunks = splitter.split_documents(docs)

    enriched_docs = []
    with st.status("🛠️ Creating Contextual Embeddings...", expanded=True) as status:
        prog = st.progress(0)

        async def process():
            for i, chunk in enumerate(chunks):
                txt = await enrich_chunk(chunk.page_content, full_text)
                enriched_docs.append(Document(page_content=txt, metadata=chunk.metadata))
                prog.progress((i + 1) / len(chunks))
                await asyncio.sleep(0.05)

        asyncio.run(process())
        status.update(label="✅ Vector Database Ready!", state="complete")

    return Chroma.from_documents(enriched_docs, embeddings)

# --- 4. CHAT INTERFACE ---
vdb = build_index()

if "messages" not in st.session_state:
    st.session_state.messages = []

# Display chat history
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# User Input
if query := st.chat_input("Ask about Large Language Models..."):
    st.session_state.messages.append({"role": "user", "content": query})
    with st.chat_message("user"):
        st.markdown(query)

    with st.chat_message("assistant"):
        # Task 3.3: Retrieve relevant chunks
        retriever = vdb.as_retriever(search_kwargs={"k": 3})
        docs = retriever.invoke(query)

        # Build prompt with context
        context_text = "\n\n---\n\n".join([d.page_content for d in docs])
        full_prompt = f"Use the following context to answer the question.\n\nContext:\n{context_text}\n\nQuestion: {query}"

        # Generate Answer
        response = llm.invoke(full_prompt)
        st.markdown(response.content)
        st.session_state.messages.append({"role": "assistant", "content": response.content})

        # Task 3.4: Display Citations
        with st.expander("📚 View Source Chunks & Metadata"):
            for i, doc in enumerate(docs):
                page_num = doc.metadata.get('page', 'Unknown')
                st.write(f"**Source {i+1} (Page {page_num+1 if isinstance(page_num, int) else page_num})**")
                st.caption(doc.page_content)

Overwriting app.py


In [10]:
from pyngrok import ngrok
import os

# 1. SET YOUR NGROK TOKEN HERE
NGROK_AUTH_TOKEN = "3BASdUOe7TKhlS8kSFXZUUZHTz8_24hWZ9Wr2mX7ZxMXn667W" # <--- PASTE TOKEN HERE
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# 2. Kill any old sessions
ngrok.kill()
!pkill -f streamlit

# 3. Start Streamlit and Open Tunnel
public_url = ngrok.connect(8501, proto="http")
print(f"🔗 YOUR CHATBOT URL: {public_url}")

# 4. Run the app
!streamlit run app.py --server.port 8501

🔗 YOUR CHATBOT URL: NgrokTunnel: "https://nonrescissible-roxanna-nonprotectively.ngrok-free.dev" -> "http://localhost:8501"



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.245.144.77:8501

Loading weights: 100% 103/103 [00:00<00:00, 2064.13it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Stopping...


In [7]:
!pkill -f streamlit
!pkill -f npx